In [1]:
import requests

url = "https://data.cityofchicago.org/resource/4ijn-s7e5.json"

response = requests.get(url, params={"$limit": 5}, timeout=30)
response.raise_for_status()

records = response.json()

print("Records returned:", len(records))
print("Type of records:", type(records))
print("Type of one record:", type(records[0]))

records[0]

Records returned: 5
Type of records: <class 'list'>
Type of one record: <class 'dict'>


{'inspection_id': '2643312',
 'dba_name': 'CROSSOVER GRILL',
 'aka_name': 'CROSSOVER GRILL',
 'license_': '3096201',
 'facility_type': 'Restaurant',
 'risk': 'Risk 3 (Low)',
 'address': '835 E 75TH ST',
 'city': 'CHICAGO',
 'state': 'IL',
 'zip': '60619',
 'inspection_date': '2026-09-24T00:00:00.000',
 'inspection_type': 'License',
 'results': 'Pass',
 'latitude': '41.75848831256902',
 'longitude': '-87.6040010137687',
 'location': {'latitude': '41.75848831256902',
  'longitude': '-87.6040010137687',
  'human_address': '{"address": "", "city": "", "state": "", "zip": ""}'},
 ':@computed_region_awaf_s7ux': '31',
 ':@computed_region_6mkv_f3dw': '21546',
 ':@computed_region_vrxf_vc4k': '67',
 ':@computed_region_bdys_3d7i': '497',
 ':@computed_region_43wa_7qmu': '35'}

In [2]:
import pandas as pd

sample = pd.DataFrame(records)
sample[["inspection_id", "facility_type", "inspection_date", "results"]]

,inspection_id,facility_type,inspection_date,results
0,2643312,Restaurant,2026-09-24T00:00:00.000,Pass
1,2643318,Restaurant,2026-09-24T00:00:00.000,Pass
2,2643249,Restaurant,2026-09-24T00:00:00.000,Pass
3,2643288,NaN,2026-09-24T00:00:00.000,Not Ready
4,2643230,Restaurant,2026-09-23T00:00:00.000,Fail


In [3]:
params = {
    "$where": "facility_type = 'Restaurant'",
    "$limit": 5
}

restaurant_response = requests.get(url, params=params, timeout=30)
restaurant_response.raise_for_status()

restaurant_records = restaurant_response.json()

pd.DataFrame(restaurant_records)[
    ["inspection_id", "facility_type", "inspection_date", "results"]
]

,inspection_id,facility_type,inspection_date,results
0,1068208,Restaurant,2012-03-14T00:00:00.000,Fail
1,1072228,Restaurant,2012-10-26T00:00:00.000,Pass
2,1072252,Restaurant,2013-01-24T00:00:00.000,Pass
3,2615511,Restaurant,2025-04-10T00:00:00.000,Pass
4,2615517,Restaurant,2025-04-10T00:00:00.000,Pass


In [4]:
where_clause = (
    "facility_type = 'Restaurant' "
    "AND inspection_date >= '2025-01-01T00:00:00' "
    "AND inspection_date < '2026-01-01T00:00:00'"
)

count_response = requests.get(
    url,
    params={"$where": where_clause, "$select": "count(*)"},
    timeout=30
)
count_response.raise_for_status()

count_response.json()

[{'count': '13864'}]

In [5]:
page_size = 1000

page_response = requests.get(
    url,
    params={
        "$where": where_clause,
        "$limit": page_size,
        "$offset": 0,
        "$order": "inspection_id ASC"
    },
    timeout=30
)
page_response.raise_for_status()

first_page = page_response.json()

print("Rows in first page:", len(first_page))
print("First ID:", first_page[0]["inspection_id"])
print("Last ID:", first_page[-1]["inspection_id"])

Rows in first page: 1000
First ID: 2609901
Last ID: 2611195


In [6]:
total_count = int(count_response.json()[0]["count"])
all_inspections = []

for offset in range(0, total_count, page_size):
    page_response = requests.get(
        url,
        params={
            "$where": where_clause,
            "$limit": page_size,
            "$offset": offset,
            "$order": "inspection_id ASC"
        },
        timeout=30
    )
    page_response.raise_for_status()

    page = page_response.json()
    all_inspections.extend(page)

    print(f"Offset {offset}: received {len(page)} records")

print("Total downloaded:", len(all_inspections))

Offset 0: received 1000 records
Offset 1000: received 1000 records
Offset 2000: received 1000 records
Offset 3000: received 1000 records
Offset 4000: received 1000 records
Offset 5000: received 1000 records
Offset 6000: received 1000 records
Offset 7000: received 1000 records
Offset 8000: received 1000 records
Offset 9000: received 1000 records
Offset 10000: received 1000 records
Offset 11000: received 1000 records
Offset 12000: received 1000 records
Offset 13000: received 864 records
Total downloaded: 13864


In [7]:
inspections = pd.DataFrame(all_inspections)

print("Shape:", inspections.shape)
print("Columns:")
print(inspections.columns.tolist())

Shape: (13864, 17)
Columns:
['inspection_id', 'dba_name', 'aka_name', 'license_', 'facility_type', 'risk', 'address', 'city', 'state', 'zip', 'inspection_date', 'inspection_type', 'results', 'latitude', 'longitude', 'location', 'violations']


In [8]:
print("Missing inspection IDs:", inspections["inspection_id"].isna().sum())
print("Duplicate inspection IDs:", inspections["inspection_id"].duplicated().sum())

print("\nFacility types:")
print(inspections["facility_type"].value_counts(dropna=False))

print("\nInspection years:")
print(inspections["inspection_date"].str[:4].value_counts(dropna=False))

Missing inspection IDs: 0
Duplicate inspection IDs: 0

Facility types:
facility_type
Restaurant    13864
Name: count, dtype: int64

Inspection years:
inspection_date
2025    13864
Name: count, dtype: int64


In [9]:
inspections.isna().sum().sort_values(ascending=False)

violations         4385
city                 17
state                14
longitude            12
location             12
latitude             12
aka_name              3
zip                   1
license_              1
inspection_id         0
address               0
dba_name              0
facility_type         0
risk                  0
results               0
inspection_date       0
inspection_type       0
dtype: int64

In [10]:
pd.crosstab(
    inspections["results"],
    inspections["violations"].isna()
)

violations,False,True
results,,
Business Not Located,0,1
Fail,2360,63
No Entry,41,1198
Not Ready,2,307
Out of Business,0,889
Pass,5322,1876
Pass w/ Conditions,1754,51


In [ ]:
from pathlib import Path

Path.cwd()

In [12]:
import json

raw_dir = Path("../data/raw")
raw_dir.mkdir(parents=True, exist_ok=True)

raw_file = raw_dir / "inspections_2025.jsonl"

with raw_file.open("w", encoding="utf-8") as file:
    for inspection in all_inspections:
        file.write(json.dumps(inspection, ensure_ascii=False) + "\n")

print(f"Saved {len(all_inspections)} inspections to {raw_file.resolve()}")

Saved 13864 inspections to ../data/raw/inspections_2025.jsonl


## Business licenses API

In [13]:
licenses_url = "https://data.cityofchicago.org/resource/r5kz-chrr.json"

license_response = requests.get(
    licenses_url,
    params={
        "$where": "license_description = 'Retail Food Establishment'",
        "$limit": 5
    },
    timeout=30
)
license_response.raise_for_status()

license_records = license_response.json()

print("Records returned:", len(license_records))
license_records[0]

Records returned: 5


{'id': '2835837-20260416',
 'license_id': '3072372',
 'account_number': '483774',
 'site_number': '1',
 'legal_name': 'DKD3 INC.',
 'doing_business_as_name': 'ROSATIS',
 'address': '1339 S HALSTED ST 1ST',
 'city': 'CHICAGO',
 'state': 'IL',
 'zip_code': '60607',
 'ward': '28',
 'precinct': '7',
 'ward_precinct': '28-7',
 'police_district': '12',
 'community_area': '28',
 'community_area_name': 'NEAR WEST SIDE',
 'neighborhood': 'LITTLE ITALY, UIC',
 'license_code': '1006',
 'license_description': 'Retail Food Establishment',
 'business_activity_id': '735',
 'business_activity': 'Preparation of Food and Dining on Premises With Seating',
 'license_number': '2835837',
 'application_type': 'RENEW',
 'application_requirements_complete': '2026-02-15T00:00:00.000',
 'payment_date': '2026-04-01T00:00:00.000',
 'conditional_approval': 'N',
 'license_start_date': '2026-04-16T00:00:00.000',
 'expiration_date': '2028-04-15T00:00:00.000',
 'license_approved_for_issuance': '2026-04-01T00:00:00.000'

In [14]:
license_where = (
    "license_description = 'Retail Food Establishment' "
    "AND license_start_date < '2026-01-01T00:00:00' "
    "AND expiration_date >= '2025-01-01T00:00:00'"
)

license_count_response = requests.get(
    licenses_url,
    params={"$where": license_where, "$select": "count(*)"},
    timeout=30
)
license_count_response.raise_for_status()

license_count_response.json()

[{'count': '17830'}]

In [15]:
license_total = int(license_count_response.json()[0]["count"])
all_licenses = []

for offset in range(0, license_total, page_size):
    page_response = requests.get(
        licenses_url,
        params={
            "$where": license_where,
            "$limit": page_size,
            "$offset": offset,
            "$order": "id ASC"
        },
        timeout=30
    )
    page_response.raise_for_status()

    page = page_response.json()
    all_licenses.extend(page)

    print(f"Offset {offset}: received {len(page)} records")

print("Total downloaded:", len(all_licenses))

Offset 0: received 1000 records
Offset 1000: received 1000 records
Offset 2000: received 1000 records
Offset 3000: received 1000 records
Offset 4000: received 1000 records
Offset 5000: received 1000 records
Offset 6000: received 1000 records
Offset 7000: received 1000 records
Offset 8000: received 1000 records
Offset 9000: received 1000 records
Offset 10000: received 1000 records
Offset 11000: received 1000 records
Offset 12000: received 1000 records
Offset 13000: received 1000 records
Offset 14000: received 1000 records
Offset 15000: received 1000 records
Offset 16000: received 1000 records
Offset 17000: received 830 records
Total downloaded: 17830


In [16]:
licenses = pd.DataFrame(all_licenses)

print("Shape:", licenses.shape)
print("Unique record IDs:", licenses["id"].nunique())
print("Unique licence numbers:", licenses["license_number"].nunique())
print("Missing licence numbers:", licenses["license_number"].isna().sum())

Shape: (17830, 37)
Unique record IDs: 17830
Unique licence numbers: 12751
Missing licence numbers: 0


In [17]:
licenses_file = raw_dir / "business_licenses_2025_overlap.jsonl"

with licenses_file.open("w", encoding="utf-8") as file:
    for license_record in all_licenses:
        file.write(json.dumps(license_record, ensure_ascii=False) + "\n")

print(f"Saved {len(all_licenses)} licence records to {licenses_file.resolve()}")

Saved 17830 licence records to ../data/raw/business_licenses_2025_overlap.jsonl


## Matching inspections to licenses

In [35]:
candidates = inspections[["inspection_id", "license_", "inspection_date"]].merge(
    licenses[["id", "license_number", "license_start_date", "expiration_date"]],
    left_on="license_",
    right_on="license_number",
    how="left"
)

print("Inspections:", len(inspections))
print("Rows after joining on license number:", len(candidates))

Inspections: 13864
Rows after joining on license number: 18948


In [36]:
candidates["inspection_date"] = pd.to_datetime(candidates["inspection_date"])
candidates["license_start_date"] = pd.to_datetime(candidates["license_start_date"])
candidates["expiration_date"] = pd.to_datetime(candidates["expiration_date"])

date_is_covered = (
    (candidates["license_start_date"] <= candidates["inspection_date"])
    & (candidates["inspection_date"] <= candidates["expiration_date"])
)
period_matches = candidates.loc[date_is_covered]

print("Rows covering the inspection date:", len(period_matches))
print("Inspections covered:", period_matches["inspection_id"].nunique())


Rows covering the inspection date: 11310
Inspections covered: 11303


In [37]:
ordered_matches = period_matches.sort_values(
    ["inspection_id", "license_start_date", "id"],
    ascending=[True, False, False]
)
chosen_matches = ordered_matches.drop_duplicates(subset="inspection_id")

print("Chosen license records:", len(chosen_matches))
print("Duplicate inspection IDs:", chosen_matches["inspection_id"].duplicated().sum())

Chosen license records: 11303
Duplicate inspection IDs: 0


In [38]:
chosen_matches = chosen_matches.rename(columns={"id": "license_record_id"})
inspection_links = inspections[["inspection_id"]].merge(
    chosen_matches[["inspection_id", "license_record_id"]],
    on="inspection_id",
    how="left"
)

print("All inspections:", len(inspection_links))
print("Duplicate inspection IDs:", inspection_links["inspection_id"].duplicated().sum())
print("Matched:", inspection_links["license_record_id"].notna().sum())
print("Unmatched:", inspection_links["license_record_id"].isna().sum())


All inspections: 13864
Duplicate inspection IDs: 0
Matched: 11303
Unmatched: 2561
